# 04 — XGBoost Forecasting Model

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np

## 1. Load data

In [ ]:
featured = pd.read_csv('../data/processed/features.csv', parse_dates=['date'])
from src.features import get_feature_columns
FEATURE_COLS = [c for c in get_feature_columns() if c in featured.columns]
TARGET = 'new_cases_7d'

cutoff = featured['date'].max() - pd.Timedelta(days=30)
train = featured[featured.date <= cutoff]
test  = featured[featured.date >  cutoff]

X_train = train[FEATURE_COLS].fillna(0).values
y_train = train[TARGET].fillna(0).values
X_test  = test[FEATURE_COLS].fillna(0).values
y_test  = test[TARGET].fillna(0).values

## 2. Train XGBoost

In [ ]:
from src.models.xgboost_model import XGBForecaster

model = XGBForecaster(n_estimators=500, max_depth=5)
model.fit(X_train, y_train,
          feature_names=FEATURE_COLS,
          eval_set=(X_test, y_test))

## 3. Evaluate

In [ ]:
from src.evaluation import evaluate
preds = model.predict(X_test)
metrics = evaluate(y_test, preds, name='XGBoost')

## 4. Feature importance & SHAP

In [ ]:
model.plot_feature_importance(top_n=15)
shap_vals = model.plot_shap(X_test, FEATURE_COLS)

## 5. Save

In [ ]:
model.save('../models/saved/xgb.pkl')